In [ ]:
from pathlib import Path
import pandas as pd
import yaml
import numpy as np

def get_project_root():
    return Path().resolve().parent

policy_encoding = pd.read_csv(get_project_root() / "data" / "raw" / "policy_encoding.csv")

path = get_project_root() / "data" / "mapping" / "index_schema.yaml"
with path.open("r", encoding="utf-8") as f:
    index_schema = yaml.safe_load(f)

def build_rename_map(schema: dict) -> dict:
    rename_map = {}

    for domain, items in schema.items():
        for key, spec in items.items():
            if isinstance(spec, dict) and "column" in spec and "indicator" in spec:
                rename_map[spec["column"]] = key

    return rename_map

# rename
rename_map = build_rename_map(index_schema)
rename_map['City/Regency'] = 'regency'
rename_map['POLICY ADOPTION'] = 'policy_version'
rename_map['Number of reported VAW cases (SIMFONI)'] = 'vaw_reported_cases'
rename_map['Gross Regional Domestic Product (GDRP) at current market prices (billion Rupiah)']='gdp'
rename_map["Adoption of regulation establishing an Integrated Service Unit for Women and Children's Protection"]='adoption_isu'
rename_map['Year'] = 'year'

policy_encoding = policy_encoding.drop(
    columns=["New city/regency year of legal establishment"]
)
policy_encoding = policy_encoding.rename(columns=rename_map)

# immediately remove domain count columns (need to recalculate, some manual errors and updates to come)
# get rid of those all-caps columns (domains)
mask = ~policy_encoding.columns.str.match(r"^[A-Z0-9\s/&\-()]+$")
policy_encoding = policy_encoding.loc[:, mask]

# map this to fullname (master)
path = get_project_root() / "data" / "mapping" / "city_mapping.yaml"
with path.open("r", encoding="utf-8") as f:
    city_mapping = yaml.safe_load(f)
city_mapping_reversed = {v: k for k, v in city_mapping.items()}

# rename and reorder columns
policy_encoding['fullname'] = policy_encoding['regency'].map(city_mapping_reversed)
policy_encoding = policy_encoding[["fullname"] + [c for c in policy_encoding.columns if c != "fullname"]]
policy_encoding = policy_encoding.drop(columns=["regency", "Province"],)

# add a column for new policy adoption (i.e. a new policy or a policy update)
policy_encoding = policy_encoding.sort_values(["fullname", "year"])
policy_encoding["policy_change"] = (
    policy_encoding.groupby("fullname")["policy_version"]
    .diff()
    .isin([1])
    .astype(int)
)
policy_encoding["has_policy"] = (policy_encoding["policy_version"] != 0).astype(int)

# add filename variable
policy_encoding["filename"] = np.select(
    [
        policy_encoding["policy_version"].eq(0),
        policy_encoding["policy_version"].eq(1),
        policy_encoding["policy_version"].eq(2),
    ],
    [
        "",
        policy_encoding["fullname"].str.replace(" ", "_") + ".txt",
        policy_encoding["fullname"].str.replace(" ", "_") + "_v2.txt",
    ],
    default=""
)

##### update raw data here ######




# save processed policy_encoding.csv
policy_encoding = policy_encoding.reset_index(drop=True)
out_path = get_project_root() / "data" / "processed" / "policy_encoding.csv"
policy_encoding.to_csv(out_path, index=False)



# calculate count columns and index data
# forward fill
# save as procecssed wide (for valudation against GDP, reporting data)


# save as processed long (for prompting)


In [60]:
policy_encoding

,fullname,year,policy_version,1.1,1.2,1.3,1.4,1.5,1.6,1.7,...,7.2,7.3,7.4,7.5,adoption_isu,vaw_reported_cases,gdp,policy_change,has_policy,filename
0,ACEH ACEH BARAT,2004,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1398.0,0,0,
1,ACEH ACEH BARAT,2005,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1317.0,0,0,
2,ACEH ACEH BARAT,2006,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1710.0,0,0,
3,ACEH ACEH BARAT,2007,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2060.0,0,0,
4,ACEH ACEH BARAT,2008,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2415.0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11193,SUMATERA UTARA TOBA,2021,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.0,8266.0,0,0,
11194,SUMATERA UTARA TOBA,2022,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,8944.0,0,0,
11195,SUMATERA UTARA TOBA,2023,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.0,9656.0,0,0,
11196,SUMATERA UTARA TOBA,2024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,10284.0,0,0,


In [58]:
# wide format, just tidy
a =policy_encoding[policy_encoding['policy_change']==1]
a[a['policy_version']>1]

,fullname,year,policy_version,1.1,1.2,1.3,1.4,1.5,1.6,1.7,...,7.2,7.3,7.4,7.5,adoption_isu,vaw_reported_cases,gdp,policy_change,has_policy,filename
1111,DI YOGYAKARTA BANTUL,2015,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,19325.0,1,1,DI_YOGYAKARTA_BANTUL_v2.txt
1138,DI YOGYAKARTA GUNUNGKIDUL,2020,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,33.0,18936.0,1,1,DI_YOGYAKARTA_GUNUNGKIDUL_v2.txt
1735,JAWA BARAT BEKASI KOTA,2023,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,7.0,118963.0,1,1,JAWA_BARAT_BEKASI_KOTA_v2.txt
2433,JAWA TENGAH KARANGANYAR,2017,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,NaN,19.0,31552.0,1,1,JAWA_TENGAH_KARANGANYAR_v2.txt
2791,JAWA TENGAH SEMARANG KOTA,2023,2,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,141.0,248909.0,1,1,JAWA_TENGAH_SEMARANG_KOTA_v2.txt
5069,KEPULAUAN BANGKA BELITUNG BANGKA BARAT,2013,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,NaN,9538.0,1,1,KEPULAUAN_BANGKA_BELITUNG_BANGKA_BARAT_v2.txt
5096,KEPULAUAN BANGKA BELITUNG BANGKA SELATAN,2018,2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,NaN,2.0,8457.0,1,1,KEPULAUAN_BANGKA_BELITUNG_BANGKA_SELATAN_v2.txt
6246,NUSA TENGGARA BARAT LOMBOK TENGAH,2024,2,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,NaN,18.0,22605.0,1,1,NUSA_TENGGARA_BARAT_LOMBOK_TENGAH_v2.txt
8377,SULAWESI SELATAN LUWU TIMUR,2021,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,14.0,23615.0,1,1,SULAWESI_SELATAN_LUWU_TIMUR_v2.txt
10336,SUMATERA SELATAN OGAN KOMERING ULU,2022,2,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,12.0,17614.0,1,1,SUMATERA_SELATAN_OGAN_KOMERING_ULU_v2.txt


In [59]:
# long format
fullname, year, filename, indicator_id, indicator_label, value

NameError: name 'fullname' is not defined